In [222]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
# train VAE to learn the dynamic of inventory control problem
# s: inventory level, a: order quantity, r: reward, o: demand
# N = 10, k = 1, c = 2, z = 2, p = 4, H = 20
# 0 \le a \le N
# o ~ N(5, 1)
# s' = max(0, min(N, s + a) - o)
# r = -k * 1(a>0) - c * (min(N, s + a) - s) - z * s + p * min(o, s+a)

class Inventory_Simulator(object):
    def __init__(self, N, k, c, z, p, H):
        self.N = N
        self.k = k
        self.c = c
        self.z = z
        self.p = p
        self.H = H

    def reset(self): # return the initial state
        return np.random.uniform(0, self.N, (1,))

    def step(self, s, a, h): # transition function
        o = np.random.normal(5, 1, (1,)) # demand
        s1 = np.clip(np.clip(s + a, a_min=None, a_max=self.N) - o, 0, None)
        # s1 = np.array(s1)
        r = -self.k * int(a > 0) - self.c * (min(self.N, s + a) - s) - self.z * s + self.p * (min(self.N, s + a)-s1)
        r = float(r)
        done = (h == self.H - 1)
        return s, a, r, s1, done
    
    def reward(self, s, a, s1):
        r = -self.k * int(a > 0) - self.c * (min(self.N, s + a) - s) - self.z * s + self.p * (min(self.N, s + a)-s1)
        return float(r)

class TransitionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(TransitionModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        mu = x[:, :1]
        sigma = x[:, 1:2]
        return mu, sigma # assume normal distribution for transition

# assume known reward function
# class RewardModel(nn.Module):
#     def __init__(self, input_dim, hidden_dim, output_dim):
#         super(RewardModel, self).__init__()
#         self.fc1 = nn.Linear(input_dim, hidden_dim)
#         self.fc2 = nn.Linear(hidden_dim, output_dim)

#     def forward(self, x):
#         x = torch.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x # assume deterministic reward

In [223]:
env = Inventory_Simulator(N=10, k=1, c=2, z=2, p=4, H=20)

In [224]:
# several [a, b] uniform policies
# several (s, S) policies
# several N(mu, sigma^2) policies
from scipy.stats import norm

class uniform_policy(object): # serve as pi_b
    def __init__(self, a, b):
        self.a = a
        self.b = b

    def act(self, s):
        a = np.random.randint(self.a, self.b+1, (1,))
        return a
    
    def prob(self, s, a):
        return 1.0 / (self.b - self.a + 1)

class approximate_uniform_policy(object): # serve as pi_e
    def __init__(self, a, b, eps):
        self.a = a
        self.b = b
        self.eps = eps
        self.idx_add_eps = range(5)
        self.idx_sub_eps = range(5, 10)
        self.prob_list = [1/11 for _ in range(11)]
        for i in self.idx_add_eps:
            self.prob_list[i] = self.prob_list[i] + self.eps
        for i in self.idx_sub_eps:
            self.prob_list[i] = self.prob_list[i] - self.eps

    def act(self, s):
        a = np.random.choice(range(11), p=self.prob_list, size=(1,))
        return a
    
    def prob(self, s, a):
        return self.prob_list[int(a)]

# class sS_policy(object):
#     def __init__(self, s, S):
#         self.a = s
#         self.b = S

#     def act(self, s):
#         if s < self.a:
#             a = self.b - s
#         else:
#             a = np.random.uniform(0, 0, (1,))
#         return a

# class normal_policy(object):
#     def __init__(self, mu, sigma):
#         self.mu = mu
#         self.sigma = sigma

#     def act(self, s):
#         # sample from 0 to 10, with the probability proportional to exp(-|x-mu+s|)
#         prob = [float(np.exp(-abs(x - self.mu + s))) for x in range(11)]
#         # normalize the probability
#         prob = [p/sum(prob) for p in prob]
#         a = np.random.choice(range(11), p=prob)
#         return a
    
#     def prob(self, s, a):
#         # calculate the probability of a given s and a
#         prob = [float(np.exp(-abs(x - self.mu + s))) for x in range(11)]
#         # normalize the probability
#         prob = [p/sum(prob) for p in prob]
#         return prob[int(a)]
#         # prob = 1.0 / (self.sigma * np.sqrt(2 * np.pi)) * np.exp(-0.5 * ((a - self.mu + s) / self.sigma) ** 2)
#         # # reweight for truncated normal distribution
#         # # compute P(0 <= X <= 10)
#         # integral_0_10 = norm.cdf(10, loc=self.mu-s, scale=self.sigma) - norm.cdf(0, loc = self.mu-s, scale=self.sigma)
#         # return prob / integral_0_10


In [263]:
pi_b = uniform_policy(0, 10) 
pi_e = approximate_uniform_policy(0, 10, 0.04)

gamma = 0.9

In [353]:
# 10 trajectories to train the model
_trajectories = []
_actions_trajectories = []
_rewards_trajectories = []
_rewards = []
for _ in range(5):
    s = env.reset()
    reward = 0
    trajectory = []
    action_trajectory = []
    reward_trajectory = []
    for h in range(env.H):
        a = pi_b.act(s)
        s, a, r, s1, done = env.step(s, a, h)
        trajectory.append(s)
        action_trajectory.append(a)
        reward_trajectory.append(r)
        reward += r * (gamma ** h)
        s = s1
    _trajectories.append(trajectory)
    _actions_trajectories.append(action_trajectory)
    _rewards_trajectories.append(reward_trajectory)
    _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.array(_trajectories),
    'actions_trajectories': np.array(_actions_trajectories),
    'rewards_trajectories': np.array(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as 0_o.pkl
with open('dr_ci_trajs/train_0_o.pkl', 'wb') as f:
    pickle.dump(data, f)

In [354]:
# 10 trajectories to calibrate
_trajectories = []
_actions_trajectories = []
_rewards_trajectories = []
_rewards = []
for _ in range(15):
    s = env.reset()
    reward = 0
    trajectory = []
    action_trajectory = []
    reward_trajectory = []
    for h in range(env.H):
        a = pi_b.act(s)
        s, a, r, s1, done = env.step(s, a, h)
        trajectory.append(s)
        action_trajectory.append(a)
        reward_trajectory.append(r)
        reward += r * (gamma ** h)
        s = s1
    _trajectories.append(trajectory)
    _actions_trajectories.append(action_trajectory)
    _rewards_trajectories.append(reward_trajectory)
    _rewards.append(reward)
# convert to dictionary
data = {
    'trajectories': np.array(_trajectories),
    'actions_trajectories': np.array(_actions_trajectories),
    'rewards_trajectories': np.array(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as 0_o.pkl
with open('dr_ci_trajs/cal_0_o.pkl', 'wb') as f:
    pickle.dump(data, f)

In [373]:
train_0_o = pickle.load(open("dr_ci_trajs/train_0_o.pkl", "rb"))
cal_0_o = pickle.load(open("dr_ci_trajs/cal_0_o.pkl", "rb"))
train_0_o_trajs = train_0_o['trajectories']
train_0_o_actions = train_0_o['actions_trajectories']
train_0_o_rewards_trajs = train_0_o['rewards_trajectories']
train_0_o_rewards = train_0_o['rewards']
cal_0_o_trajs = cal_0_o['trajectories']
cal_0_o_actions = cal_0_o['actions_trajectories']
cal_0_o_rewards_trajs = cal_0_o['rewards_trajectories']
cal_0_o_rewards = cal_0_o['rewards']

# merge the two datasets
whole_0_o_trajs = np.concatenate((train_0_o_trajs, cal_0_o_trajs), axis=0)
whole_0_o_actions = np.concatenate((train_0_o_actions, cal_0_o_actions), axis=0)
whole_0_o_rewards_trajs = np.concatenate((train_0_o_rewards_trajs, cal_0_o_rewards_trajs), axis=0)
whole_0_o_rewards = np.concatenate((train_0_o_rewards, cal_0_o_rewards), axis=0)

In [374]:
whole_weights = np.zeros(len(whole_0_o_trajs))
for i in range(len(whole_0_o_trajs)):
    w = 1
    for t in range(len(whole_0_o_trajs[i])):
        w *= pi_e.prob(whole_0_o_trajs[i][t], whole_0_o_actions[i][t]) / pi_b.prob(whole_0_o_trajs[i][t], whole_0_o_actions[i][t])
    whole_weights[i] = w

In [375]:
alpha = 0.2
z = norm.ppf(1 - alpha/2)

In [376]:
# IS CLT
V_IS = np.zeros(len(whole_0_o_trajs))
for i in range(len(whole_0_o_trajs)):
    V_IS[i] = whole_weights[i] * (whole_0_o_rewards[i])
V_IS_mean = np.mean(V_IS)
V_IS_var = np.var(V_IS)
V_IS_lb = V_IS_mean - z * np.sqrt(V_IS_var / len(V_IS))
V_IS_ub = V_IS_mean + z * np.sqrt(V_IS_var / len(V_IS))
print('V_IS:', V_IS_mean)
print('V_IS_LB:', V_IS_lb)
print('V_IS_UB:', V_IS_ub)

V_IS: 16.491360728084295
V_IS_LB: 7.690415322922641
V_IS_UB: 25.29230613324595


In [377]:
# DR CLT

# train a DQN using the batch data
class DQN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
Q = DQN(2, 16, 1)
# make batch data (s, a, r, s1)
batch_data = []
for i in range(len(train_0_o_trajs)):
    for t in range(len(train_0_o_trajs[i]) - 1):
        s = train_0_o_trajs[i][t]
        a = train_0_o_actions[i][t]
        r = train_0_o_rewards_trajs[i][t]
        s1 = train_0_o_trajs[i][t + 1]
        batch_data.append((s, a, r, s1))

optimizer = optim.Adam(Q.parameters(), lr=0.001)
criterion = nn.MSELoss()
num_epochs = 5000
batch_size = 64
for epoch in range(num_epochs):
    np.random.shuffle(batch_data)
    for i in range(0, len(batch_data), batch_size):
        batch = batch_data[i:i + batch_size]
        s = torch.tensor([b[0] for b in batch], dtype=torch.float32)
        a = torch.tensor([b[1] for b in batch], dtype=torch.float32)
        r = torch.tensor([b[2] for b in batch], dtype=torch.float32)
        s1 = torch.tensor([b[3] for b in batch], dtype=torch.float32)
       
        q_value = Q(torch.cat((s, a), dim=1))
        V_pi_e = 0
        for a in range(11):
            prob = torch.tensor([pi_e.prob(s1, a)]).unsqueeze(0).expand(s1.shape[0], 1)
            a = torch.tensor(a, dtype=torch.float32)
            a = a.unsqueeze(0).expand(s1.shape[0], 1)
            V_pi_e += prob * Q(torch.cat((s1, a), dim=1))
        
        target_q_value = r.unsqueeze(1) + gamma * V_pi_e
        loss = criterion(q_value, target_q_value)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 100 == 0:
        print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.4f}')

V_DR_step = np.zeros(len(cal_0_o_trajs))
for i in range(len(cal_0_o_trajs)):
    v = 0
    for t in range(len(cal_0_o_trajs[i])):
        s = cal_0_o_trajs[i][t]
        a = cal_0_o_actions[i][t]
        r = cal_0_o_rewards_trajs[i][t]
        V_s = -100
        
        tensor_s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
        tensor_a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
        for a1 in range(11):
            tensor_a1 = torch.tensor([a1], dtype=torch.float32).unsqueeze(0)
            V_s_a1 = Q(torch.cat((tensor_s, tensor_a1), dim=1)).detach().numpy()
            V_s_a1 = float(V_s_a1)
            V_s = max(V_s, V_s_a1)
        Q_s_a = Q(torch.cat((tensor_s, tensor_a), dim=1)).detach().numpy()
        Q_s_a = float(Q_s_a)
        w = pi_e.prob(s, a) / pi_b.prob(s, a)
        v = V_s + w * (r + gamma * v - Q_s_a)
    V_DR_step[i] = v

# calculate the average of V_DR
V_DR_step_mean = np.mean(V_DR_step)
V_DR_step_var = np.var(V_DR_step)
V_DR_step_lb = V_DR_step_mean - z * np.sqrt(V_DR_step_var / len(V_DR_step))
V_DR_step_ub = V_DR_step_mean + z * np.sqrt(V_DR_step_var / len(V_DR_step))
print('V_DR:', V_DR_step_mean)
print('V_DR_LB:', V_DR_step_lb)
print('V_DR_UB:', V_DR_step_ub)

Epoch [0/5000], Loss: 12.7661
Epoch [100/5000], Loss: 22.8768
Epoch [200/5000], Loss: 20.7880
Epoch [300/5000], Loss: 14.5151
Epoch [400/5000], Loss: 12.0549
Epoch [500/5000], Loss: 17.2276
Epoch [600/5000], Loss: 15.6664
Epoch [700/5000], Loss: 14.3185
Epoch [800/5000], Loss: 16.8186
Epoch [900/5000], Loss: 15.3556
Epoch [1000/5000], Loss: 11.0972
Epoch [1100/5000], Loss: 9.4011
Epoch [1200/5000], Loss: 10.2885
Epoch [1300/5000], Loss: 13.1066
Epoch [1400/5000], Loss: 13.0432
Epoch [1500/5000], Loss: 18.3092
Epoch [1600/5000], Loss: 10.1286
Epoch [1700/5000], Loss: 6.7863
Epoch [1800/5000], Loss: 13.1093
Epoch [1900/5000], Loss: 9.9348
Epoch [2000/5000], Loss: 11.3430
Epoch [2100/5000], Loss: 14.3945
Epoch [2200/5000], Loss: 17.5856
Epoch [2300/5000], Loss: 11.7655
Epoch [2400/5000], Loss: 8.6178
Epoch [2500/5000], Loss: 13.3678
Epoch [2600/5000], Loss: 11.4458
Epoch [2700/5000], Loss: 9.4929
Epoch [2800/5000], Loss: 10.4054
Epoch [2900/5000], Loss: 11.5656
Epoch [3000/5000], Loss: 11

In [378]:
# IS Bootstrap
bootstrap_samples = len(whole_0_o_trajs) * 10
V_IS_bootstrap = np.zeros(bootstrap_samples)
for i in range(bootstrap_samples):
    idx = np.random.randint(0, len(whole_0_o_trajs), size=len(whole_0_o_trajs))
    V_IS_bootstrap[i] = np.mean(V_IS[idx])
V_IS_bootstrap_mean = np.mean(V_IS_bootstrap)
# get the alpha quantile and (1-alpha) quantile
V_IS_bootstrap_alpha = np.quantile(V_IS_bootstrap, alpha)
V_IS_bootstrap_1_alpha = np.quantile(V_IS_bootstrap, 1 - alpha)
print('V_IS_bootstrap:', V_IS_bootstrap_mean)
print('V_IS_bootstrap_lb:', V_IS_bootstrap_alpha)
print('V_IS_bootstrap_1_ub:', V_IS_bootstrap_1_alpha)


V_IS_bootstrap: 16.2824727354562
V_IS_bootstrap_lb: 10.46833402876062
V_IS_bootstrap_1_ub: 21.974822945964373


In [379]:
# train a transition model
def train_T(data, epochs, batch_size):
    batch_data = []
    train_0_o_trajs = data['trajectories']
    train_0_o_actions = data['actions_trajectories']
    train_0_o_rewards_trajs = data['rewards_trajectories']
    train_0_o_rewards = data['rewards']
    for i in range(len(train_0_o_trajs)):
        for t in range(len(train_0_o_trajs[i]) - 1):
            s = train_0_o_trajs[i][t]
            a = train_0_o_actions[i][t]
            r = train_0_o_rewards_trajs[i][t]
            s1 = train_0_o_trajs[i][t + 1]
            batch_data.append((s, a, r, s1))
            
    T = TransitionModel(input_dim=2, hidden_dim=64, output_dim=2) # input: (s, a), output: (mu, sigma)
    optimizer_T = optim.Adam(T.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    num_epochs = epochs
    batch_size = batch_size

    for epoch in range(num_epochs):
        # shuffle the data
        np.random.shuffle(batch_data)
        for i in range(0, len(batch_data), batch_size):
            batch = batch_data[i:i + batch_size]
            s = torch.tensor([b[0] for b in batch], dtype=torch.float32)
            a = torch.tensor([b[1] for b in batch], dtype=torch.float32)
            r = torch.tensor([b[2] for b in batch], dtype=torch.float32)
            s1 = torch.tensor([b[3] for b in batch], dtype=torch.float32)

            optimizer_T.zero_grad()
            mu, sigma = T(torch.cat((s, a), dim=1))
            s1_pred = torch.randn(s1.shape) * sigma + mu
            loss_T = criterion(s1_pred, s1)
            loss_T.backward()
            optimizer_T.step()

        if epoch % int(num_epochs / 10) == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Loss T: {loss_T.item():.4f}')
    return T

In [381]:
# MB Bootstrap
bootstrap_samples = 10
V_MB_bootstrap = np.zeros(bootstrap_samples)
for i in range(bootstrap_samples):
    # sample 20 trajectories from whole_0_o_trajs
    idx = np.random.randint(0, len(whole_0_o_trajs), len(whole_0_o_trajs))
    data = {
        'trajectories': whole_0_o_trajs[idx],
        'actions_trajectories': whole_0_o_actions[idx],
        'rewards_trajectories': whole_0_o_rewards_trajs[idx],
        'rewards': whole_0_o_rewards[idx]
    }
    T = train_T(data, epochs=1000, batch_size=32)
    # rollout 1000 trajectories
    V = 0
    for j in range(1000):
        s = env.reset() # known initial state distribution
        reward = 0
        for h in range(env.H):
            a = pi_e.act(s)
            s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
            a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
            mu, sigma = T(torch.cat((s, a), dim=1))
            s1 = torch.randn(s.shape) * sigma + mu
            r = env.reward(s.detach().numpy(), a.detach().numpy(), s1.detach().numpy())
            reward += r * (gamma ** h)
            s1 = s1.detach().numpy().reshape(1,)
            s = s1
        V += reward
    V_MB_bootstrap[i] = V / 1000

Epoch [0/1000], Loss T: 8.1446
Epoch [100/1000], Loss T: 0.9002
Epoch [200/1000], Loss T: 0.8881
Epoch [300/1000], Loss T: 0.6709
Epoch [400/1000], Loss T: 1.1915
Epoch [500/1000], Loss T: 0.2739
Epoch [600/1000], Loss T: 0.5963
Epoch [700/1000], Loss T: 0.8856
Epoch [800/1000], Loss T: 0.4684
Epoch [900/1000], Loss T: 0.4435
Epoch [0/1000], Loss T: 6.2713
Epoch [100/1000], Loss T: 1.0463
Epoch [200/1000], Loss T: 0.4171
Epoch [300/1000], Loss T: 0.6204
Epoch [400/1000], Loss T: 0.4289
Epoch [500/1000], Loss T: 0.7110
Epoch [600/1000], Loss T: 0.6080
Epoch [700/1000], Loss T: 0.7590
Epoch [800/1000], Loss T: 1.0868
Epoch [900/1000], Loss T: 0.8223
Epoch [0/1000], Loss T: 23.0878
Epoch [100/1000], Loss T: 0.8844
Epoch [200/1000], Loss T: 0.5790
Epoch [300/1000], Loss T: 0.7214
Epoch [400/1000], Loss T: 0.2216
Epoch [500/1000], Loss T: 0.4558
Epoch [600/1000], Loss T: 0.7429
Epoch [700/1000], Loss T: 0.7615
Epoch [800/1000], Loss T: 1.0895
Epoch [900/1000], Loss T: 0.6404
Epoch [0/1000],

In [382]:
V_MB_bootstrap_mean = np.mean(V_MB_bootstrap)
# get the alpha quantile and (1-alpha) quantile
V_MB_bootstrap_alpha = np.quantile(V_MB_bootstrap, alpha)
V_MB_bootstrap_1_alpha = np.quantile(V_MB_bootstrap, 1 - alpha)
print('V_MB_bootstrap:', V_MB_bootstrap_mean)
print('V_MB_bootstrap_lb:', V_MB_bootstrap_alpha)
print('V_MB_bootstrap_ub:', V_MB_bootstrap_1_alpha)

V_MB_bootstrap: 27.592060949377686
V_MB_bootstrap_lb: 26.152342963078834
V_MB_bootstrap_ub: 28.399128502527592


In [383]:
# train a transition model using whole_0_o
whole_0_o = {
    'trajectories': whole_0_o_trajs,
    'actions_trajectories': whole_0_o_actions,
    'rewards_trajectories': whole_0_o_rewards_trajs,
    'rewards': whole_0_o_rewards
}
T = train_T(whole_0_o, epochs=1000, batch_size=64)

Epoch [0/1000], Loss T: 18.8475
Epoch [100/1000], Loss T: 1.1112
Epoch [200/1000], Loss T: 0.7530
Epoch [300/1000], Loss T: 0.7456
Epoch [400/1000], Loss T: 0.9127
Epoch [500/1000], Loss T: 0.7308
Epoch [600/1000], Loss T: 0.6978
Epoch [700/1000], Loss T: 0.5221
Epoch [800/1000], Loss T: 0.4610
Epoch [900/1000], Loss T: 0.4519


In [384]:
# generate augmented data
augmeng_samples = 5000
_trajectories = []
_actions_trajectories = []
_rewards_trajectories = []
_rewards = []

for i in range(augmeng_samples):
    s = env.reset()
    reward = 0
    trajectory = []
    action_trajectory = []
    reward_trajectory = []
    for h in range(env.H):
        a = pi_b.act(s)
        trajectory.append(s)
        action_trajectory.append(a)
        s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
        a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
        mu, sigma = T(torch.cat((s, a), dim=1))
        s1 = torch.randn(s.shape) * sigma + mu
        r = env.reward(s.detach().numpy(), a.detach().numpy(), s1.detach().numpy())
        reward_trajectory.append(r)
        reward += r * (gamma ** h)
        s1 = s1.detach().numpy().reshape(1,)
        s = s1
    _trajectories.append(trajectory)
    _actions_trajectories.append(action_trajectory)
    _rewards.append(reward)
    _rewards_trajectories.append(reward_trajectory)
# convert to dictionary
data = {
    'trajectories': np.array(_trajectories),
    'actions_trajectories': np.array(_actions_trajectories),
    'rewards_trajectories': np.array(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as 0_o.pkl
with open('dr_ci_trajs/aug_0_diff.pkl', 'wb') as f:
    pickle.dump(data, f)

In [385]:
aug_0_diff = pickle.load(open("dr_ci_trajs/aug_0_diff.pkl", "rb"))
aug_0_diff_trajs = aug_0_diff['trajectories']
aug_0_diff_actions = aug_0_diff['actions_trajectories']
aug_0_diff_rewards_trajs = aug_0_diff['rewards_trajectories']
aug_0_diff_rewards = aug_0_diff['rewards']

In [386]:
# augment IS CLT
aug_weights = np.zeros(len(aug_0_diff_trajs))
for i in range(len(aug_0_diff_trajs)):
    w = 1
    for t in range(len(aug_0_diff_trajs[i])):
        w *= pi_e.prob(aug_0_diff_trajs[i][t], aug_0_diff_actions[i][t]) / pi_b.prob(aug_0_diff_trajs[i][t], aug_0_diff_actions[i][t])
    aug_weights[i] = w

V_IS_aug = np.zeros(len(aug_0_diff_trajs))
for i in range(len(aug_0_diff_trajs)):
    V_IS_aug[i] = aug_weights[i] * (aug_0_diff_rewards[i])
V_IS_aug_mean = np.mean(V_IS_aug)
V_IS_aug_var = np.var(V_IS_aug)
V_IS_aug_lb = V_IS_aug_mean - z * np.sqrt(V_IS_aug_var / len(V_IS_aug))
V_IS_aug_ub = V_IS_aug_mean + z * np.sqrt(V_IS_aug_var / len(V_IS_aug))
print('V_IS_aug:', V_IS_aug_mean)
print('V_IS_aug_LB:', V_IS_aug_lb)
print('V_IS_aug_UB:', V_IS_aug_ub)

V_IS_aug: 31.126434636668606
V_IS_aug_LB: 27.37783664221562
V_IS_aug_UB: 34.87503263112159


In [388]:
# augment DR CLT

# DR CLT
aug_Q = DQN(2, 16, 1)

# seperate the data aug_0_diff into train and test (1:4)
train_0_diff_trajs = aug_0_diff_trajs[:int(len(aug_0_diff_trajs) * 0.2)]
train_0_diff_actions = aug_0_diff_actions[:int(len(aug_0_diff_trajs) * 0.2)]
train_0_diff_rewards_trajs = aug_0_diff_rewards_trajs[:int(len(aug_0_diff_trajs) * 0.2)]
train_0_diff_rewards = aug_0_diff_rewards[:int(len(aug_0_diff_trajs) * 0.2)]
cal_0_diff_trajs = aug_0_diff_trajs[int(len(aug_0_diff_trajs) * 0.2):]
cal_0_diff_actions = aug_0_diff_actions[int(len(aug_0_diff_trajs) * 0.2):]
cal_0_diff_rewards_trajs = aug_0_diff_rewards_trajs[int(len(aug_0_diff_trajs) * 0.2):]
cal_0_diff_rewards = aug_0_diff_rewards[int(len(aug_0_diff_trajs) * 0.2):]


# make batch data (s, a, r, s1)
batch_data = []
for i in range(len(train_0_diff_trajs)):
    for t in range(len(train_0_diff_trajs[i]) - 1):
        s = train_0_diff_trajs[i][t]
        a = train_0_diff_actions[i][t]
        r = train_0_diff_rewards_trajs[i][t]
        s1 = train_0_diff_trajs[i][t + 1]
        batch_data.append((s, a, r, s1))

optimizer = optim.Adam(aug_Q.parameters(), lr=0.001)
criterion = nn.MSELoss()
num_epochs = 500
batch_size = 64
for epoch in range(num_epochs):
    np.random.shuffle(batch_data)
    for i in range(0, len(batch_data), batch_size):
        batch = batch_data[i:i + batch_size]
        s = torch.tensor([b[0] for b in batch], dtype=torch.float32)
        a = torch.tensor([b[1] for b in batch], dtype=torch.float32)
        r = torch.tensor([b[2] for b in batch], dtype=torch.float32)
        s1 = torch.tensor([b[3] for b in batch], dtype=torch.float32)
       
        q_value = aug_Q(torch.cat((s, a), dim=1))
        V_pi_e = 0
        for a in range(11):
            prob = torch.tensor([pi_e.prob(s1, a)]).unsqueeze(0).expand(s1.shape[0], 1)
            a = torch.tensor(a, dtype=torch.float32)
            a = a.unsqueeze(0).expand(s1.shape[0], 1)
            V_pi_e += prob * aug_Q(torch.cat((s1, a), dim=1))
        
        target_q_value = r.unsqueeze(1) + gamma * V_pi_e
        loss = criterion(q_value, target_q_value)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if epoch % 100 == 0:
        print(f'Epoch [{epoch}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [0/500], Loss: 12.5236
Epoch [100/500], Loss: 0.5808
Epoch [200/500], Loss: 0.8559
Epoch [300/500], Loss: 0.3676
Epoch [400/500], Loss: 0.2071


In [389]:
V_DR_step_aug = np.zeros(len(cal_0_diff_trajs))
for i in range(len(cal_0_diff_trajs)):
    v = 0
    for t in range(len(cal_0_diff_trajs[i])):
        s = cal_0_diff_trajs[i][t]
        a = cal_0_diff_actions[i][t]
        r = cal_0_diff_rewards_trajs[i][t]
        V_s = -100
        
        tensor_s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
        tensor_a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
        for a1 in range(11):
            tensor_a1 = torch.tensor([a1], dtype=torch.float32).unsqueeze(0)
            V_s_a1 = aug_Q(torch.cat((tensor_s, tensor_a1), dim=1)).detach().numpy()
            V_s_a1 = float(V_s_a1)
            V_s = max(V_s, V_s_a1)
        Q_s_a = aug_Q(torch.cat((tensor_s, tensor_a), dim=1)).detach().numpy()
        Q_s_a = float(Q_s_a)
        w = pi_e.prob(s, a) / pi_b.prob(s, a)
        v = V_s + w * (r + gamma * v - Q_s_a)
    V_DR_step_aug[i] = v

# calculate the average of V_DR_step_aug
V_DR_step_aug_mean = np.mean(V_DR_step_aug)
V_DR_step_aug_var = np.var(V_DR_step_aug)
V_DR_step_aug_lb = V_DR_step_aug_mean - z * np.sqrt(V_DR_step_aug_var / len(V_DR_step_aug))
V_DR_step_aug_ub = V_DR_step_aug_mean + z * np.sqrt(V_DR_step_aug_var / len(V_DR_step_aug))
print('V_DR_step_aug:', V_DR_step_aug_mean)
print('V_DR_step_aug_LB:', V_DR_step_aug_lb)
print('V_DR_step_aug_UB:', V_DR_step_aug_ub)

V_DR_step_aug: 65.83724148375451
V_DR_step_aug_LB: 65.26321783291374
V_DR_step_aug_UB: 66.41126513459528


In [390]:
# augment IS Bootstrap
bootstrap_samples = 200
V_IS_bootstrap_aug = np.zeros(bootstrap_samples)
for i in range(bootstrap_samples):
    idx = np.random.randint(0, len(aug_0_diff_trajs), size=len(aug_0_diff_trajs))
    V_IS_bootstrap_aug[i] = np.mean(V_IS_aug[idx])
V_IS_bootstrap_aug_mean = np.mean(V_IS_bootstrap_aug)
# get the alpha quantile and (1-alpha) quantile
V_IS_bootstrap_aug_alpha = np.quantile(V_IS_bootstrap_aug, alpha)
V_IS_bootstrap_aug_1_alpha = np.quantile(V_IS_bootstrap_aug, 1 - alpha)
print('V_IS_bootstrap_aug:', V_IS_bootstrap_aug_mean)
print('V_IS_bootstrap_aug_lb:', V_IS_bootstrap_aug_alpha)
print('V_IS_bootstrap_aug_ub:', V_IS_bootstrap_aug_1_alpha)

V_IS_bootstrap_aug: 31.133483447415788
V_IS_bootstrap_aug_lb: 28.615293591186443
V_IS_bootstrap_aug_ub: 33.37655401610647


In [393]:
# augment MB Bootstrap
bootstrap_samples = 10
V_MB_bootstrap_aug = np.zeros(bootstrap_samples)
for i in range(bootstrap_samples):
    idx = np.random.randint(0, len(aug_0_diff_trajs), len(aug_0_diff_trajs))
    data = {
        'trajectories': aug_0_diff_trajs[idx],
        'actions_trajectories': aug_0_diff_actions[idx],
        'rewards_trajectories': aug_0_diff_rewards_trajs[idx],
        'rewards': aug_0_diff_rewards[idx]
    }
    T = train_T(data, epochs=50, batch_size=256)
    # rollout 1000 trajectories
    V = 0
    for j in range(1000):
        s = env.reset() # known initial state distribution
        reward = 0
        for h in range(env.H):
            a = pi_e.act(s)
            s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
            a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
            mu, sigma = T(torch.cat((s, a), dim=1))
            s1 = torch.randn(s.shape) * sigma + mu
            r = env.reward(s.detach().numpy(), a.detach().numpy(), s1.detach().numpy()) # known reward function
            reward += r * (gamma ** h)
            s1 = s1.detach().numpy().reshape(1,)
            s = s1
        V += reward
    V_MB_bootstrap_aug[i] = V / 1000
V_MB_bootstrap_aug_mean = np.mean(V_MB_bootstrap_aug)
# get the alpha quantile and (1-alpha) quantile
V_MB_bootstrap_aug_alpha = np.quantile(V_MB_bootstrap_aug, alpha)
V_MB_bootstrap_aug_1_alpha = np.quantile(V_MB_bootstrap_aug, 1 - alpha)
print('V_MB_bootstrap_aug:', V_MB_bootstrap_aug_mean)
print('V_MB_bootstrap_aug_lb:', V_MB_bootstrap_aug_alpha)
print('V_MB_bootstrap_aug_ub:', V_MB_bootstrap_aug_1_alpha)

Epoch [0/50], Loss T: 0.1489
Epoch [5/50], Loss T: 0.0258
Epoch [10/50], Loss T: 0.0039
Epoch [15/50], Loss T: 0.0016
Epoch [20/50], Loss T: 0.0028
Epoch [25/50], Loss T: 0.0015
Epoch [30/50], Loss T: 0.0015
Epoch [35/50], Loss T: 0.0035
Epoch [40/50], Loss T: 0.0009
Epoch [45/50], Loss T: 0.0023
Epoch [0/50], Loss T: 0.2657
Epoch [5/50], Loss T: 0.0136
Epoch [10/50], Loss T: 0.0071
Epoch [15/50], Loss T: 0.0054
Epoch [20/50], Loss T: 0.0018
Epoch [25/50], Loss T: 0.0031
Epoch [30/50], Loss T: 0.0015
Epoch [35/50], Loss T: 0.0029
Epoch [40/50], Loss T: 0.0019
Epoch [45/50], Loss T: 0.0024
Epoch [0/50], Loss T: 0.2091
Epoch [5/50], Loss T: 0.0508
Epoch [10/50], Loss T: 0.0036
Epoch [15/50], Loss T: 0.0036
Epoch [20/50], Loss T: 0.0015
Epoch [25/50], Loss T: 0.0019
Epoch [30/50], Loss T: 0.0026
Epoch [35/50], Loss T: 0.0025
Epoch [40/50], Loss T: 0.0015
Epoch [45/50], Loss T: 0.0016
Epoch [0/50], Loss T: 0.2698
Epoch [5/50], Loss T: 0.0195
Epoch [10/50], Loss T: 0.0035
Epoch [15/50], Los

In [359]:
# DR-PPI
# train a transition model using train_0_o
T = train_T(train_0_o, epochs=500, batch_size=16)

Epoch [0/500], Loss T: 7.8986
Epoch [50/500], Loss T: 0.7917
Epoch [100/500], Loss T: 1.0183
Epoch [150/500], Loss T: 0.6608
Epoch [200/500], Loss T: 0.4321
Epoch [250/500], Loss T: 0.7000
Epoch [300/500], Loss T: 0.8406
Epoch [350/500], Loss T: 0.3681
Epoch [400/500], Loss T: 0.2935
Epoch [450/500], Loss T: 0.7602


In [360]:
# generate 5000 trajectories, serve as the first term in DR-PPI N_f=5000
N_f = 1000
_trajectories = []
_actions_trajectories = []
_rewards_trajectories = []
_rewards = []
for i in range(N_f):
    s = env.reset()
    reward = 0
    trajectory = []
    action_trajectory = []
    reward_trajectory = []
    for h in range(env.H):
        a = pi_e.act(s)
        trajectory.append(s)
        action_trajectory.append(a)
        s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
        a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
        mu, sigma = T(torch.cat((s, a), dim=1))
        s1 = torch.randn(s.shape) * sigma + mu
        r = env.reward(s, a, s1)
        reward_trajectory.append(r)
        reward += r * (gamma ** h)
        s1 = s1.detach().numpy().reshape(1,)
        s = s1
    _trajectories.append(trajectory)
    _actions_trajectories.append(action_trajectory)
    _rewards.append(reward)
    _rewards_trajectories.append(reward_trajectory)
# convert to dictionary
data = {
    'trajectories': np.array(_trajectories),
    'actions_trajectories': np.array(_actions_trajectories),
    'rewards_trajectories': np.array(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as first_term_DRPPI.pkl
with open('dr_ci_trajs/first_term_DRPPI.pkl', 'wb') as f:
    pickle.dump(data, f)

In [361]:
# for each traj in cal_0_o_trajs, generate 100 trajectories M=100
M = 100
_trajectories = []
_actions_trajectories = []
_rewards_trajectories = []
_rewards = []
for i in range(len(cal_0_o_trajs)):
    s0 = cal_0_o_trajs[i][0]
    for _ in range(M):
        s = s0
        reward = 0
        trajectory = []
        action_trajectory = []
        reward_trajectory = []
        for h in range(env.H):
            a = pi_e.act(s)
            trajectory.append(s)
            action_trajectory.append(a)
            s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
            a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
            mu, sigma = T(torch.cat((s, a), dim=1))
            s1 = torch.randn(s.shape) * sigma + mu
            r = env.reward(s, a, s1)
            reward_trajectory.append(r)
            reward += r * (gamma ** h)
            s1 = s1.detach().numpy().reshape(1,)
            s = s1
        _trajectories.append(trajectory)
        _actions_trajectories.append(action_trajectory)
        _rewards.append(reward)
        _rewards_trajectories.append(reward_trajectory)
# convert to dictionary
data = {
    'trajectories': np.array(_trajectories),
    'actions_trajectories': np.array(_actions_trajectories),
    'rewards_trajectories': np.array(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as second_term_matching_DRPPI.pkl
with open('dr_ci_trajs/second_term_matching_DRPPI.pkl', 'wb') as f:
    pickle.dump(data, f)



In [362]:
# second term for two PPI methods
second_term_matching_DRPPI = pickle.load(open("dr_ci_trajs/second_term_matching_DRPPI.pkl", "rb"))
second_term_matching_DRPPI_trajs = second_term_matching_DRPPI['trajectories']
second_term_matching_DRPPI_actions = second_term_matching_DRPPI['actions_trajectories']
second_term_matching_DRPPI_rewards_trajs = second_term_matching_DRPPI['rewards_trajectories']
second_term_matching_DRPPI_rewards = second_term_matching_DRPPI['rewards']

cal_s0_pairs = []
for i in range(len(cal_0_o_trajs)):
    for j in range(len(second_term_matching_DRPPI_trajs)):
        if cal_0_o_trajs[i][0] == second_term_matching_DRPPI_trajs[j][0]:
            cal_s0_pairs.append((i, j))

print(len(cal_s0_pairs))

1500


In [363]:
# DR-PPI
# first term for DR-PPI methods
first_term_DRPPI = pickle.load(open("dr_ci_trajs/first_term_DRPPI.pkl", "rb"))
first_term_DRPPI_trajs = first_term_DRPPI['trajectories']
first_term_DRPPI_actions = first_term_DRPPI['actions_trajectories']
first_term_DRPPI_rewards_trajs = first_term_DRPPI['rewards_trajectories']
first_term_DRPPI_rewards = first_term_DRPPI['rewards']
first_term_DRPPI_rewards_mean = np.mean(first_term_DRPPI_rewards)
sigma_f2 = np.var(first_term_DRPPI_rewards)

#IS weights
weights = np.zeros(len(cal_0_o_trajs))
for i in range(len(cal_0_o_trajs)):
    w = 1
    for t in range(len(cal_0_o_trajs[i])):
        w *= pi_e.prob(cal_0_o_trajs[i][t], cal_0_o_actions[i][t]) / pi_b.prob(cal_0_o_trajs[i][t], cal_0_o_actions[i][t])
    weights[i] = w

E_diff_rewards_IS = np.zeros(len(cal_0_o_trajs))
for i, j in cal_s0_pairs:  
    diff = weights[i]*cal_0_o_rewards[i] - second_term_matching_DRPPI_rewards[j]
    E_diff_rewards_IS[i] += diff
E_diff_rewards_IS = E_diff_rewards_IS / M

E_diff_rewards_IS_mean = np.mean(E_diff_rewards_IS)
sigma_b2_IS = np.var(E_diff_rewards_IS)

# WIS
normalized_weights = np.zeros(len(cal_0_o_trajs))
normalized_weights = weights / np.sum(weights)
E_diff_rewards_WIS = np.zeros(len(cal_0_o_trajs))
for i, j in cal_s0_pairs:  
    diff = normalized_weights[i]*cal_0_o_rewards[i] - second_term_matching_DRPPI_rewards[j]
    E_diff_rewards_WIS[i] += diff
E_diff_rewards_WIS = E_diff_rewards_WIS / M
E_diff_rewards_WIS_mean = np.mean(E_diff_rewards_WIS)
sigma_b2_WIS = np.var(E_diff_rewards_WIS)

# PDIS
reweighted_values = np.zeros(len(cal_0_o_trajs))
for i in range(len(cal_0_o_trajs)):
    w = 1
    v = 0
    for t in range(len(cal_0_o_trajs[i])):
        w *= pi_e.prob(cal_0_o_trajs[i][t], cal_0_o_actions[i][t]) / pi_b.prob(cal_0_o_trajs[i][t], cal_0_o_actions[i][t])
        v += cal_0_o_rewards_trajs[i][t] * (gamma ** t) * w
    reweighted_values[i] = v

E_diff_rewards_PDIS = np.zeros(len(cal_0_o_trajs))
for i, j in cal_s0_pairs:  
    diff = reweighted_values[i] - second_term_matching_DRPPI_rewards[j]
    E_diff_rewards_PDIS[i] += diff
E_diff_rewards_PDIS = E_diff_rewards_PDIS / M
E_diff_rewards_PDIS_mean = np.mean(E_diff_rewards_PDIS)
sigma_b2_PDIS = np.var(E_diff_rewards_PDIS)

V_DRPPI_IS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_IS_mean
V_DRPPI_IS_var = sigma_f2/N_f + sigma_b2_IS/len(cal_0_o_trajs)
V_DRPPI_IS_lb = V_DRPPI_IS_mean - z * np.sqrt(V_DRPPI_IS_var)
V_DRPPI_IS_ub = V_DRPPI_IS_mean + z * np.sqrt(V_DRPPI_IS_var)
print('V_DRPPI_IS:', V_DRPPI_IS_mean)
print('V_DRPPI_IS_LB:', V_DRPPI_IS_lb)
print('V_DRPPI_IS_UB:', V_DRPPI_IS_ub)

V_DRPPI_WIS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_WIS_mean
V_DRPPI_WIS_var = sigma_f2/N_f + sigma_b2_WIS/len(cal_0_o_trajs)
V_DRPPI_WIS_lb = V_DRPPI_WIS_mean - z * np.sqrt(V_DRPPI_WIS_var)
V_DRPPI_WIS_ub = V_DRPPI_WIS_mean + z * np.sqrt(V_DRPPI_WIS_var)
print('V_DRPPI_WIS:', V_DRPPI_WIS_mean)
print('V_DRPPI_WIS_LB:', V_DRPPI_WIS_lb)
print('V_DRPPI_WIS_UB:', V_DRPPI_WIS_ub)

V_DRPPI_PDIS_mean = first_term_DRPPI_rewards_mean + E_diff_rewards_PDIS_mean
V_DRPPI_PDIS_var = sigma_f2/N_f + sigma_b2_PDIS/len(cal_0_o_trajs)
V_DRPPI_PDIS_lb = V_DRPPI_PDIS_mean - z * np.sqrt(V_DRPPI_PDIS_var)
V_DRPPI_PDIS_ub = V_DRPPI_PDIS_mean + z * np.sqrt(V_DRPPI_PDIS_var)
print('V_DRPPI_PDIS:', V_DRPPI_PDIS_mean)
print('V_DRPPI_PDIS_LB:', V_DRPPI_PDIS_lb)
print('V_DRPPI_PDIS_UB:', V_DRPPI_PDIS_ub)

V_DRPPI_IS: 24.482738281545632
V_DRPPI_IS_LB: 12.280952799086478
V_DRPPI_IS_UB: 36.68452376400479
V_DRPPI_WIS: 5.0991102797121854
V_DRPPI_WIS_LB: 2.8617912552106626
V_DRPPI_WIS_UB: 7.336429304213708
V_DRPPI_PDIS: 45.00475884747178
V_DRPPI_PDIS_LB: 22.643596277863285
V_DRPPI_PDIS_UB: 67.36592141708027


In [364]:
# CP-PPI
# resplit the data into train and test (1:1)
train_0_o_trajs = whole_0_o_trajs[:int(len(whole_0_o_trajs) * 0.5)]
train_0_o_actions = whole_0_o_actions[:int(len(whole_0_o_trajs) * 0.5)]
train_0_o_rewards_trajs = whole_0_o_rewards_trajs[:int(len(whole_0_o_trajs) * 0.5)]
train_0_o_rewards = whole_0_o_rewards[:int(len(whole_0_o_trajs) * 0.5)]
cal_0_o_trajs = whole_0_o_trajs[int(len(whole_0_o_trajs) * 0.5):]
cal_0_o_actions = whole_0_o_actions[int(len(whole_0_o_trajs) * 0.5):]
cal_0_o_rewards_trajs = whole_0_o_rewards_trajs[int(len(whole_0_o_trajs) * 0.5):]
cal_0_o_rewards = whole_0_o_rewards[int(len(whole_0_o_trajs) * 0.5):]
# train a transition model using train_0_o
T = train_T(train_0_o, epochs=500, batch_size=16)

Epoch [0/500], Loss T: 5.9729
Epoch [50/500], Loss T: 0.8716
Epoch [100/500], Loss T: 0.9326
Epoch [150/500], Loss T: 1.0880
Epoch [200/500], Loss T: 0.8508
Epoch [250/500], Loss T: 0.6025
Epoch [300/500], Loss T: 0.6489
Epoch [350/500], Loss T: 1.0420
Epoch [400/500], Loss T: 0.9266
Epoch [450/500], Loss T: 0.5018


In [365]:
# for each traj in train_0_o_trajs, generate M trajectories
M = 100
_trajectories = []
_actions_trajectories = []
_rewards_trajectories = []
_rewards = []
for i in range(len(train_0_o_trajs)):
    s0 = train_0_o_trajs[i][0]
    for _ in range(M):
        s = s0
        reward = 0
        trajectory = []
        action_trajectory = []
        reward_trajectory = []
        for h in range(env.H):
            a = pi_b.act(s)
            trajectory.append(s)
            action_trajectory.append(a)
            s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
            a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
            mu, sigma = T(torch.cat((s, a), dim=1))
            s1 = torch.randn(s.shape) * sigma + mu
            r = env.reward(s, a, s1)
            reward_trajectory.append(r)
            reward += r * (gamma ** h)
            s1 = s1.detach().numpy().reshape(1,)
            s = s1
        _trajectories.append(trajectory)
        _actions_trajectories.append(action_trajectory)
        _rewards.append(reward)
        _rewards_trajectories.append(reward_trajectory)
# convert to dictionary
data = {
    'trajectories': np.array(_trajectories),
    'actions_trajectories': np.array(_actions_trajectories),
    'rewards_trajectories': np.array(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as second_term_matching_DRPPI.pkl
with open('dr_ci_trajs/train_matching_CPPPI.pkl', 'wb') as f:
    pickle.dump(data, f)



In [366]:
# for each traj in cal_0_o_trajs, generate M trajectories
M = 100
_trajectories = []
_actions_trajectories = []
_rewards_trajectories = []
_rewards = []
for i in range(len(cal_0_o_trajs)):
    s0 = cal_0_o_trajs[i][0]
    for _ in range(M):
        s = s0
        reward = 0
        trajectory = []
        action_trajectory = []
        reward_trajectory = []
        for h in range(env.H):
            a = pi_b.act(s)
            trajectory.append(s)
            action_trajectory.append(a)
            s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
            a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
            mu, sigma = T(torch.cat((s, a), dim=1))
            s1 = torch.randn(s.shape) * sigma + mu
            r = env.reward(s, a, s1)
            reward_trajectory.append(r)
            reward += r * (gamma ** h)
            s1 = s1.detach().numpy().reshape(1,)
            s = s1
        _trajectories.append(trajectory)
        _actions_trajectories.append(action_trajectory)
        _rewards.append(reward)
        _rewards_trajectories.append(reward_trajectory)
# convert to dictionary
data = {
    'trajectories': np.array(_trajectories),
    'actions_trajectories': np.array(_actions_trajectories),
    'rewards_trajectories': np.array(_rewards_trajectories),
    'rewards': np.array(_rewards)
}
# save as second_term_matching_DRPPI.pkl
with open('dr_ci_trajs/cal_matching_CPPPI.pkl', 'wb') as f:
    pickle.dump(data, f)



In [367]:
# CP-PPI
train_matching_PPI = pickle.load(open("dr_ci_trajs/train_matching_CPPPI.pkl", "rb"))
train_matching_PPI_trajs = train_matching_PPI['trajectories']
train_matching_PPI_actions = train_matching_PPI['actions_trajectories']
train_matching_PPI_rewards_trajs = train_matching_PPI['rewards_trajectories']
train_matching_PPI_rewards = train_matching_PPI['rewards']

cal_matching_PPI = pickle.load(open("dr_ci_trajs/cal_matching_CPPPI.pkl", "rb"))
cal_matching_PPI_trajs = cal_matching_PPI['trajectories']
cal_matching_PPI_actions = cal_matching_PPI['actions_trajectories']
cal_matching_PPI_rewards_trajs = cal_matching_PPI['rewards_trajectories']
cal_matching_PPI_rewards = cal_matching_PPI['rewards']

train_s0_pairs = []
for i in range(len(train_0_o_trajs)):
    for j in range(len(train_matching_PPI_trajs)):
        if train_0_o_trajs[i][0] == train_matching_PPI_trajs[j][0]:
            train_s0_pairs.append((i, j))

cal_s0_pairs = []
for i in range(len(cal_0_o_trajs)):
    for j in range(len(cal_matching_PPI_trajs)):
        if cal_0_o_trajs[i][0] == cal_matching_PPI_trajs[j][0]:
            cal_s0_pairs.append((i, j))

train_diff_rewards = []
for i, j in train_s0_pairs:
    diff = train_0_o_rewards[i] - train_matching_PPI_rewards[j]
    train_diff_rewards.append((i, j, diff)) # state = train_0_o_trajs[i]

cal_diff_rewards = []
for i, j in cal_s0_pairs:
    diff = cal_0_o_rewards[i] - cal_matching_PPI_rewards[j]
    cal_diff_rewards.append((i, j, diff)) # state = cal_0_o_trajs[i]

In [368]:
len(cal_diff_rewards), len(train_diff_rewards)

(1000, 1000)

In [369]:
eps_s = 2
eps_r = 30

matching_pairs = []
for m in range(len(cal_diff_rewards)):
    i, j, diff = cal_diff_rewards[m]
    for n in range(len(train_diff_rewards)):
        i2, j2, diff2 = train_diff_rewards[n]
        if abs(cal_0_o_trajs[i][0] - train_0_o_trajs[i2][0]) < eps_s and \
            abs(diff - diff2) < eps_r:
            matching_pairs.append((m, n)) # train_n can be used to calculate the weights for cal_m
print(len(matching_pairs))

377606


In [370]:
weights = []
for m in range(len(cal_diff_rewards)):
    w = 0
    n = 0
    for m1, n1 in matching_pairs:
        w1 = 1
        if m == m1:
            i2, j2, diff2 = train_diff_rewards[n1]
            for t in range(len(train_0_o_trajs[i2])):
                w1 *= pi_e.prob(train_0_o_trajs[i2][t], train_0_o_actions[i2][t]) * pi_e.prob(train_matching_PPI_trajs[j2][t], train_matching_PPI_actions[j2][t]) / \
                    pi_b.prob(train_0_o_trajs[i2][t], train_0_o_actions[i2][t]) / pi_b.prob(train_matching_PPI_trajs[j2][t], train_matching_PPI_actions[j2][t])
            w += w1
            n += 1
    if n==0:
        print('no matching pairs')
        i, j, diff = cal_diff_rewards[m]
        print(cal_0_o_trajs[i][0], cal_matching_PPI_trajs[j][0], diff)
        break
    weights.append(w/n)

# sort the weights and delta_reward
weights = np.array(weights, dtype=np.float32)
delta_reward = np.array([diff for _, _, diff in cal_diff_rewards], dtype=np.float32)
delta_reward_sorted = np.sort(delta_reward)
weights_sorted = weights[np.argsort(delta_reward)]

In [371]:
def w(x, y, eps_s, eps_r):
    matching_pairs = []
    for i1, j1 in train_s0_pairs:
        if abs(train_0_o_trajs[i1][0] - x) < eps_s and \
            abs((train_0_o_rewards[i1] - train_matching_PPI_rewards[j1]) - y) < eps_r:
            matching_pairs.append((i1, j1))
    w = 0
    n = len(matching_pairs)
    for i, j in matching_pairs:
        w1 = 1
        for t in range(len(train_0_o_trajs[i])):
            w1 *= pi_e.prob(train_0_o_trajs[i][t], train_0_o_actions[i][t]) * pi_e.prob(train_matching_PPI_trajs[j][t], train_matching_PPI_actions[j][t]) / \
                pi_b.prob(train_0_o_trajs[i][t], train_0_o_actions[i][t]) / pi_b.prob(train_matching_PPI_trajs[j][t], train_matching_PPI_actions[j][t])
        w += w1
    if n > 0:
        return w/n
    else:
        return -1
    
start_s = 5
y_list = np.linspace(-100, 100, 100)
CP_s_y = []
for y in y_list:
    w_ = w(start_s, y, eps_s, eps_r)
    if w_ != -1:
        new_weights = np.append(weights_sorted, w_)
        normalized_weights = new_weights / np.sum(new_weights)
        
        low_level = 0
        high_level = 0
        low_q = 0
        high_q = 0
        for i in range(len(delta_reward_sorted)):
            low_level += normalized_weights[i]
            if low_level >= alpha/2:
                low_q = delta_reward_sorted[i-1]
                break
        for i in range(len(delta_reward_sorted)):
            high_level += normalized_weights[i]
            if high_level >= 1 - alpha/2:
                high_q = delta_reward_sorted[i-1]
                break
        print("y: ", y, "low_q: ", low_q, "high_q: ", high_q)
        break
        # if low_q <= y <= high_q:
        #     CP_1_y.append(y)

y:  -71.71717171717171 low_q:  -4.567647 high_q:  38.77841


In [372]:
_rewards = []
for _ in range(500):
    s = np.array(start_s).reshape(1,)
    reward = 0
    trajectory = []
    action_trajectory = []
    for h in range(env.H):
        a = pi_e.act(s)
        trajectory.append(s)
        action_trajectory.append(a)
        s = torch.tensor(s, dtype=torch.float32).unsqueeze(0)
        a = torch.tensor(a, dtype=torch.float32).unsqueeze(0)
        mu, sigma = T(torch.cat((s, a), dim=1))
        s1 = torch.randn(s.shape) * sigma + mu
        r = env.reward(s, a, s1)
        reward += r * (gamma ** h)
        s1 = s1.detach().numpy().reshape(1,)
        s = s1
    _trajectories.append(trajectory)
    _actions_trajectories.append(action_trajectory)
    _rewards.append(reward)

hat_V_pi_e = np.mean(_rewards)
CP = (hat_V_pi_e + low_q, hat_V_pi_e + high_q)
print("CP: ", CP)

CP:  (18.867588101001942, 62.21364694652196)


In [395]:
# evaluate the policies by MC for 10000 episodes
def evaluate_policy(env, policy, gamma, num_episodes=1000):
    total_reward = 0
    for _ in range(num_episodes):
        # s = env.reset()
        s = np.array(start_s).reshape(1,)
        for h in range(env.H):
            a = policy.act(s)
            s, a, r, s1, done = env.step(s, a, h)
            total_reward += r * (gamma ** h)
            s = s1
    return total_reward / num_episodes

evaluate_policy(env, pi_e, gamma=0.9, num_episodes=10000)

28.43803449232428